## encodings: CAMELYON17-WILDS x UNI2-h / Virchow2

Runs CAMELYON17-WILDS's 96x96 lymph-node patches through [UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h) and [Virchow2](https://huggingface.co/paige-ai/Virchow2), and saves patch encodings plus their tumor/center/split metadata.

The full release is ~450k patches -- encoding all of it is a valid (larger) run on a GPU box, but for the course workflow this notebook draws a stratified sample by `(center, tumor)` so a full pass finishes quickly. Set `SAMPLE_SIZE = None` to embed everything.

In [ ]:
!pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import os
import sys
from pathlib import Path

os.environ["HF_HOME"] = "/home/shared/.cache/huggingface"
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

sys.path.append("/home/shared/helper/")

from vfm_encoders import embed_images, load_uni2, load_virchow2

In [ ]:
# Check if CUDA (GPU support) is available
is_available = torch.cuda.is_available()
print(f"GPU Available: {is_available}")

# Get the number of available GPUs
device_count = torch.cuda.device_count()
print(f"Number of GPUs: {device_count}")

if is_available:
    # Get the name of the current GPU
    print(f"Current GPU Name: {torch.cuda.get_device_name(0)}")

### 0. Setup

Both encoders are gated on Hugging Face -- request access on the model pages, then set `HF_TOKEN` (or leave unset for an interactive prompt).

In [ ]:
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print(
        "WARNING: no GPU found -- these are large ViT-H encoders, CPU is only "
        "fine for a quick smoke test on a handful of patches."
    )

### 1. Load metadata and draw a sample

Same `metadata_df` construction as `exploration/explore_camelyon.ipynb`, then a stratified sample so the encoding run stays fast.

In [ ]:
camelyon_base_dir = "/home/shared/data/camelyon17/camelyon17_v1.0/"
metadata_path = os.path.join(camelyon_base_dir, "metadata.csv")

metadata_df = (
    pd.read_csv(metadata_path, index_col=False)
    .drop("Unnamed: 0", axis=1)
    .assign(
        patient_node=lambda df_: df_.apply(
            lambda row: f"patient_{row['patient']:03d}_node_{row['node']}", axis=1
        )
    )
    .assign(
        filepath=lambda df_: df_.apply(
            lambda row: Path(
                f"{row['patient_node']}/patch_{row['patient_node']}_x_{row['x_coord']}_y_{row['y_coord']}.png"
            ),
            axis=1,
        )
    )
    .assign(
        filepath_abs=lambda df_: df_["filepath"].apply(
            lambda p: Path(camelyon_base_dir) / "patches" / p
        )
    )
    .drop("patient_node", axis=1)
)

In [ ]:
SAMPLE_SIZE = 100  # set to None to embed the full ~450k-patch release

if SAMPLE_SIZE is not None:
    metadata_df = metadata_df.groupby(["center", "tumor"]).sample(SAMPLE_SIZE, random_state=42)

print(f"Patches selected: {len(metadata_df):,}")
metadata_df["tumor"].value_counts()

### 2. Load the encoders

In [ ]:
encoders = {
    "uni2-h": load_uni2(device),
    "virchow2": load_virchow2(device),
}
for enc in encoders.values():
    n_params = sum(p.numel() for p in enc.model.parameters())
    print(f"{enc.name:10s} embed_dim={enc.embed_dim:5d}  params={n_params / 1e6:.0f}M")

### 3. Load patches and extract encodings

Decodes each PNG once into memory, then runs both encoders over the same list of patches -- avoids re-reading disk twice for two encoders.

In [ ]:
patches = [
    Image.open(p).convert("RGB") for p in tqdm(metadata_df["filepath_abs"], desc="loading patches")
]

encodings = {enc.name: embed_images(enc, patches, device=device) for enc in encoders.values()}

### 4. Save encodings

One `.npz` of features per model (row order matches `metadata_df`), plus the metadata itself as a CSV so a probing notebook can join on row index.

In [ ]:
out_dir = Path.home() / "data" / "camelyon17" / "encodings"
out_dir.mkdir(parents=True, exist_ok=True)

suffix = f"n{len(metadata_df)}"
for model_name, feats in encodings.items():
    out_path = out_dir / f"{model_name}_{suffix}.npy"
    np.save(out_path, feats)
    print("Saved:", out_path)

metadata_out_path = out_dir / f"metadata_{suffix}.csv"
metadata_df.drop(columns=["filepath_abs"]).to_csv(metadata_out_path, index=False)
print("Saved:", metadata_out_path)

### 5. Sanity check

PCA down to 2D, colored by tumor label and by center -- CAMELYON17's whole premise is inter-center variability, so a good encoder should separate tumor/normal more than it separates by hospital.

In [ ]:
from sklearn.decomposition import PCA

fig, axs = plt.subplots(2, len(encoders), figsize=(6 * len(encoders), 10))
for col, model_name in enumerate(encodings):
    coords = PCA(n_components=2, random_state=42).fit_transform(encodings[model_name])

    ax = axs[0, col]
    for tumor, group in metadata_df.assign(x=coords[:, 0], y=coords[:, 1]).groupby("tumor"):
        ax.scatter(group["x"], group["y"], s=8, alpha=0.5, label=f"tumor={tumor}")
    ax.set_title(f"{model_name} -- by tumor")
    ax.legend(fontsize=8)

    ax = axs[1, col]
    for center, group in metadata_df.assign(x=coords[:, 0], y=coords[:, 1]).groupby("center"):
        ax.scatter(group["x"], group["y"], s=8, alpha=0.5, label=f"center={center}")
    ax.set_title(f"{model_name} -- by center")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()